In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="11-people-you-may-know",
    notebook_name="02_graph_based_approaches.ipynb"
)

# 02 -- Graph-Based Approaches for People You May Know

**Goal:** Master the graph algorithms and neural network approaches that power PYMK systems.

---

## The Friendship Map Analogy

Imagine your entire school's friendships drawn on a giant poster. Each student is a dot, and every friendship is a line connecting two dots. This poster IS a **social graph**.

Now, to predict who should be friends, you could:
1. **Count mutual friends** (simple but powerful -- like seeing two dots with many shared connections)
2. **Walk randomly through the map** (DeepWalk/Node2Vec -- like wandering the halls and seeing who you bump into)
3. **Use a neural network that understands maps** (GNN -- a brain that can read the entire friendship poster)

This notebook covers all three approaches with working code.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
from collections import defaultdict
import math
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

## 1. Social Graph Representations

### Three Ways to Store a Graph

| Representation | Space | Edge Lookup | Iterate Neighbors | Best For |
|---------------|-------|-------------|-------------------|----------|
| **Adjacency Matrix** | O(n^2) | O(1) | O(n) | Dense graphs, matrix operations |
| **Adjacency List** | O(n + m) | O(degree) | O(degree) | Sparse graphs (social networks!) |
| **Edge List** | O(m) | O(m) | O(m) | Storage, data transfer |

In [ ]:
# === Build a social network with community structure ===

def build_social_network(n_users=80, n_clusters=4, intra_prob=0.3, inter_prob=0.01):
    """
    Build a synthetic social network using a stochastic block model.
    
    12-year-old version:
    4 friend groups at school. Within each group, 30% chance of friendship.
    Between groups, only 1% chance. Just like real life!
    """
    G = nx.Graph()
    cluster_names = ['Engineering', 'Marketing', 'Research', 'Sales']
    
    cluster_sizes = [n_users // n_clusters] * n_clusters
    cluster_sizes[-1] += n_users - sum(cluster_sizes)
    
    cluster_labels = {}
    uid = 0
    for cidx, size in enumerate(cluster_sizes):
        for _ in range(size):
            G.add_node(uid, cluster=cidx, cluster_name=cluster_names[cidx])
            cluster_labels[uid] = cidx
            uid += 1
    
    for i in range(n_users):
        for j in range(i + 1, n_users):
            p = intra_prob if cluster_labels[i] == cluster_labels[j] else inter_prob
            if random.random() < p:
                G.add_edge(i, j)
    
    return G, cluster_labels


G, cluster_labels = build_social_network()

print(f"Social Network Statistics")
print(f"=" * 40)
print(f"Nodes (users):     {G.number_of_nodes()}")
print(f"Edges (friendships): {G.number_of_edges()}")
print(f"Avg degree:         {2 * G.number_of_edges() / G.number_of_nodes():.1f}")
print(f"Density:            {nx.density(G):.4f}")
print(f"Components:         {nx.number_connected_components(G)}")

# Show all three representations
print(f"\n--- Adjacency Matrix (first 6x6) ---")
adj_matrix = nx.to_numpy_array(G)
print(adj_matrix[:6, :6].astype(int))

print(f"\n--- Adjacency List (first 3 users) ---")
for node in range(3):
    neighbors = list(G.neighbors(node))
    print(f"  User {node}: {neighbors}")

print(f"\n--- Edge List (first 5 edges) ---")
for i, (u, v) in enumerate(list(G.edges())[:5]):
    print(f"  ({u}, {v})")

# Memory comparison
n = G.number_of_nodes()
m = G.number_of_edges()
print(f"\n--- Memory Comparison ---")
print(f"  Adjacency Matrix: {n*n*8/1024:.1f} KB (O(n^2) = {n*n:,} entries)")
print(f"  Adjacency List:   {(n + 2*m)*8/1024:.1f} KB (O(n+m))")
print(f"  Edge List:        {2*m*8/1024:.1f} KB (O(m))")
print(f"\n  For LinkedIn (1B users): adjacency matrix = {1e9*1e9*8/1e18:.0f} exabytes!")
print(f"  Adjacency list is the only practical choice.")

In [ ]:
# Visualize the network with communities
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
node_colors = [colors[cluster_labels[n]] for n in G.nodes()]
pos = nx.spring_layout(G, seed=42, k=0.6)

# Left: full network
nx.draw(G, pos, ax=axes[0], node_color=node_colors, node_size=100,
        edge_color='#cccccc', width=0.3, alpha=0.9)
for i, name in enumerate(['Engineering', 'Marketing', 'Research', 'Sales']):
    axes[0].scatter([], [], c=colors[i], s=80, label=name)
axes[0].legend(title='Department', fontsize=9)
axes[0].set_title('Social Network with Community Structure', fontsize=12)

# Right: adjacency matrix heatmap
# Sort by cluster for block structure
sorted_nodes = sorted(G.nodes(), key=lambda n: cluster_labels[n])
sorted_adj = nx.to_numpy_array(G, nodelist=sorted_nodes)
im = axes[1].imshow(sorted_adj, cmap='Blues', aspect='auto')
axes[1].set_xlabel('User ID (sorted by department)', fontsize=11)
axes[1].set_ylabel('User ID (sorted by department)', fontsize=11)
axes[1].set_title('Adjacency Matrix (sorted by community)\nBlock structure = communities!', fontsize=12)
plt.colorbar(im, ax=axes[1], label='Connected')

# Add cluster boundaries
sizes = [sum(1 for v in cluster_labels.values() if v == c) for c in range(4)]
cum = 0
for s in sizes[:-1]:
    cum += s
    axes[1].axhline(cum - 0.5, color='red', linewidth=1, alpha=0.5)
    axes[1].axvline(cum - 0.5, color='red', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

print("The block structure in the adjacency matrix reveals communities.")
print("Dense blocks on the diagonal = within-department friendships.")
print("Sparse off-diagonal = cross-department friendships (rare).")

## 2. Graph Feature Engineering

These are the hand-crafted features that measure how "close" two non-connected users are in the graph.

### The Big Three + Extras

| Feature | Formula | Intuition |
|---------|---------|----------|
| **Common Neighbors (CN)** | |N(u) intersection N(v)| | How many mutual friends? |
| **Jaccard Coefficient** | CN / |N(u) union N(v)| | What fraction of combined friends are shared? |
| **Adamic-Adar** | sum(1/log(degree(w))) for w in CN | Picky mutual friends count more |
| **Preferential Attachment** | degree(u) * degree(v) | Popular people make more connections |
| **Resource Allocation** | sum(1/degree(w)) for w in CN | Similar to AA but sharper penalty |

In [ ]:
def compute_all_graph_features(G, u, v):
    """
    Compute comprehensive graph features for a user pair.
    
    12-year-old version:
    Imagine you are trying to figure out if two kids should be introduced.
    You check:
    - How many friends they SHARE (Common Neighbors)
    - What FRACTION of their friends overlap (Jaccard)
    - Whether their shared friends are picky or not (Adamic-Adar)
    - How popular each kid is (Preferential Attachment)
    - How connected their neighborhoods are (Resource Allocation)
    """
    N_u = set(G.neighbors(u))
    N_v = set(G.neighbors(v))
    common = N_u & N_v
    union = N_u | N_v
    
    cn = len(common)
    jaccard = cn / len(union) if union else 0.0
    
    adamic_adar = sum(1.0 / math.log(G.degree(w)) for w in common if G.degree(w) > 1)
    
    pref_attach = G.degree(u) * G.degree(v)
    
    resource_alloc = sum(1.0 / G.degree(w) for w in common if G.degree(w) > 0)
    
    return {
        'common_neighbors': cn,
        'jaccard': round(jaccard, 4),
        'adamic_adar': round(adamic_adar, 4),
        'preferential_attachment': pref_attach,
        'resource_allocation': round(resource_alloc, 4),
        'degree_u': G.degree(u),
        'degree_v': G.degree(v),
    }


# Get FoF candidates for a target user
def get_fof_candidates(G, user):
    """Get ranked FoF candidates."""
    friends = set(G.neighbors(user))
    fof = defaultdict(int)
    for f in friends:
        for fof_candidate in G.neighbors(f):
            if fof_candidate != user and fof_candidate not in friends:
                fof[fof_candidate] += 1
    return sorted(fof.items(), key=lambda x: -x[1])


target = 0
candidates = get_fof_candidates(G, target)

print(f"Graph Features for User {target}'s Top FoF Candidates")
print(f"({'=' * 80})")
print(f"{'Cand':>5} {'CN':>4} {'Jaccard':>8} {'AA':>8} {'PA':>6} {'RA':>8} {'Cluster':>12}")
print("-" * 55)
for cand, _ in candidates[:12]:
    f = compute_all_graph_features(G, target, cand)
    cluster = G.nodes[cand]['cluster_name']
    print(f"{cand:>5} {f['common_neighbors']:>4} {f['jaccard']:>8.4f} {f['adamic_adar']:>8.4f} "
          f"{f['preferential_attachment']:>6} {f['resource_allocation']:>8.4f} {cluster:>12}")

In [ ]:
# Visualize: How different features rank candidates differently

all_features = []
for cand, _ in candidates:
    f = compute_all_graph_features(G, target, cand)
    f['candidate'] = cand
    f['same_cluster'] = int(cluster_labels[cand] == cluster_labels[target])
    all_features.append(f)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('common_neighbors', 'Common Neighbors', 'Raw count of mutual friends'),
    ('jaccard', 'Jaccard Coefficient', 'Fraction of combined friends shared'),
    ('adamic_adar', 'Adamic-Adar Index', 'Picky mutual friends weighted more'),
    ('resource_allocation', 'Resource Allocation', 'Similar to AA, sharper penalty'),
]

for idx, (metric, title, subtitle) in enumerate(metrics):
    ax = axes[idx // 2][idx % 2]
    vals = [f[metric] for f in all_features]
    same = [f['same_cluster'] for f in all_features]
    cols = ['#FF6B6B' if s else '#45B7D1' for s in same]
    
    sorted_idx = np.argsort(vals)[::-1]
    sorted_vals = [vals[i] for i in sorted_idx]
    sorted_cols = [cols[i] for i in sorted_idx]
    
    ax.barh(range(min(20, len(sorted_vals))), sorted_vals[:20], color=sorted_cols[:20], alpha=0.8)
    ax.set_xlabel(title, fontsize=10)
    ax.set_ylabel('Rank', fontsize=10)
    ax.set_title(f'{title}\n({subtitle})', fontsize=11)
    ax.invert_yaxis()

axes[0][0].scatter([], [], c='#FF6B6B', s=60, label='Same department')
axes[0][0].scatter([], [], c='#45B7D1', s=60, label='Different department')
axes[0][0].legend(fontsize=9)

plt.suptitle(f'Candidate Rankings by Different Graph Features (User {target})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: Same-cluster candidates (red) dominate the top ranks")
print("across ALL metrics. Graph features naturally capture community structure.")

## 3. Node2Vec / DeepWalk Embeddings

### The Idea

Instead of hand-crafting graph features, what if we could learn a **vector representation** for each node that captures its position and role in the graph?

**DeepWalk/Node2Vec** does this by:
1. Performing random walks on the graph (like wandering through the school)
2. Treating each walk as a "sentence" (sequence of nodes)
3. Using Word2Vec (Skip-gram) to learn embeddings from these "sentences"

```
Random Walk: [Alice] -> [Bob] -> [Charlie] -> [Diana] -> [Eve]
                       (This is like a "sentence" in NLP!)
                       
Word2Vec learns: users who appear in similar walks get similar embeddings.
```

**Node2Vec** extends DeepWalk with two parameters:
- **p (return parameter):** Controls likelihood of backtracking. Low p = explore locally (like BFS)
- **q (in-out parameter):** Controls likelihood of moving away. Low q = explore broadly (like DFS)

In [ ]:
class SimpleNode2Vec:
    """
    Simplified Node2Vec implementation for educational purposes.
    
    12-year-old version:
    Imagine you drop a ball on the friendship map. It bounces from
    person to person randomly. You record where it goes. Do this
    many times from every person. People whose balls follow similar
    paths are probably in similar positions in the network.
    
    Staff-level detail:
    - Node2Vec uses biased random walks controlled by p (return) and q (in-out)
    - p < 1: more likely to return (BFS-like, captures local structure)
    - q < 1: more likely to explore (DFS-like, captures global structure)
    - Walks are fed to Word2Vec Skip-gram to learn embeddings
    - Time complexity: O(walks_per_node * walk_length * n)
    """
    
    def __init__(self, G, embed_dim=32, walk_length=20, walks_per_node=10,
                 p=1.0, q=1.0, window_size=5):
        self.G = G
        self.embed_dim = embed_dim
        self.walk_length = walk_length
        self.walks_per_node = walks_per_node
        self.p = p
        self.q = q
        self.window_size = window_size
        self.embeddings = {}
    
    def random_walk(self, start_node):
        """Perform a single biased random walk."""
        walk = [start_node]
        neighbors = list(self.G.neighbors(start_node))
        if not neighbors:
            return walk
        
        walk.append(random.choice(neighbors))
        
        for _ in range(self.walk_length - 2):
            cur = walk[-1]
            prev = walk[-2]
            neighbors = list(self.G.neighbors(cur))
            if not neighbors:
                break
            
            # Compute biased weights
            weights = []
            for nbr in neighbors:
                if nbr == prev:
                    weights.append(1.0 / self.p)   # Return to previous
                elif self.G.has_edge(nbr, prev):
                    weights.append(1.0)              # Stay in neighborhood
                else:
                    weights.append(1.0 / self.q)   # Move away
            
            total = sum(weights)
            weights = [w / total for w in weights]
            next_node = random.choices(neighbors, weights=weights)[0]
            walk.append(next_node)
        
        return walk
    
    def generate_walks(self):
        """Generate all random walks."""
        walks = []
        nodes = list(self.G.nodes())
        for _ in range(self.walks_per_node):
            random.shuffle(nodes)
            for node in nodes:
                walks.append(self.random_walk(node))
        return walks
    
    def learn_embeddings_simple(self):
        """
        Learn node embeddings using a simplified Skip-gram approach.
        For production, you'd use gensim Word2Vec.
        """
        walks = self.generate_walks()
        
        # Count co-occurrences within window
        n_nodes = self.G.number_of_nodes()
        cooccurrence = np.zeros((n_nodes, n_nodes))
        
        for walk in walks:
            for i, center in enumerate(walk):
                start = max(0, i - self.window_size)
                end = min(len(walk), i + self.window_size + 1)
                for j in range(start, end):
                    if i != j:
                        cooccurrence[center][walk[j]] += 1
        
        # Use SVD on the co-occurrence matrix for embeddings
        # (In practice, Skip-gram with negative sampling is better)
        cooccurrence_log = np.log1p(cooccurrence)
        U, S, Vt = np.linalg.svd(cooccurrence_log, full_matrices=False)
        self.embeddings = U[:, :self.embed_dim] * np.sqrt(S[:self.embed_dim])
        
        return self.embeddings


# Train Node2Vec embeddings
print("Training Node2Vec embeddings...")
n2v = SimpleNode2Vec(G, embed_dim=16, walk_length=15, walks_per_node=10, p=1.0, q=1.0)
embeddings = n2v.learn_embeddings_simple()

print(f"\nNode2Vec Embeddings")
print(f"=" * 40)
print(f"Nodes: {len(embeddings)}")
print(f"Embedding dim: {embeddings.shape[1]}")
print(f"\nSample embedding for user 0:")
print(f"  [{', '.join(f'{x:.3f}' for x in embeddings[0][:8])}...]")

In [ ]:
# Visualize embeddings with t-SNE
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=15)
embeddings_2d = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: colored by cluster
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
for cidx in range(4):
    mask = [cluster_labels[n] == cidx for n in range(G.number_of_nodes())]
    axes[0].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   c=colors[cidx], s=40, alpha=0.7, 
                   label=['Engineering', 'Marketing', 'Research', 'Sales'][cidx])

axes[0].legend(fontsize=10)
axes[0].set_title('Node2Vec Embeddings (t-SNE)\nColors = departments', fontsize=12)
axes[0].set_xlabel('t-SNE 1', fontsize=11)
axes[0].set_ylabel('t-SNE 2', fontsize=11)

# Right: colored by degree
degrees = [G.degree(n) for n in range(G.number_of_nodes())]
sc = axes[1].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                    c=degrees, cmap='YlOrRd', s=40, alpha=0.7)
plt.colorbar(sc, ax=axes[1], label='Degree (# connections)')
axes[1].set_title('Node2Vec Embeddings (t-SNE)\nColors = node degree', fontsize=12)
axes[1].set_xlabel('t-SNE 1', fontsize=11)
axes[1].set_ylabel('t-SNE 2', fontsize=11)

plt.tight_layout()
plt.show()

print("Left: Node2Vec correctly clusters users by department!")
print("The embeddings learned community structure from random walks alone.")
print("\nRight: High-degree nodes (hubs) tend to be in distinct positions.")

## 4. Graph Neural Networks (GNNs)

### The Upgrade from Node2Vec to GNNs

Node2Vec learns embeddings from **structure only** (who is connected to whom). GNNs learn from **both structure AND features** (who is connected AND what their profiles look like).

### How GNNs Work (Message Passing)

Think of it like a game of telephone:
1. Each person starts with their own info (age, company, school)
2. Round 1: Everyone shares their info with their direct friends
3. Round 2: Everyone shares their UPDATED info (which now includes friends' info)
4. After K rounds, each person knows about their K-hop neighborhood

This is called **message passing** and it is the core of GNNs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleGCNLayer(nn.Module):
    """
    Simplified Graph Convolutional Network (GCN) layer.
    
    12-year-old version:
    Each person collects their friends' info, averages it together
    with their own info, and creates a new summary. After doing this
    a few times, each person's summary captures not just their own
    info but their entire neighborhood's info.
    
    Staff-level detail:
    h_v^{(l+1)} = sigma(W * MEAN(h_u^{(l)} for u in N(v) union {v}))
    
    GCN vs GraphSAGE vs GAT:
    - GCN: simple mean aggregation, full-batch training
    - GraphSAGE: SAMPLES neighbors, supports mini-batch (scales better)
    - GAT: uses ATTENTION weights on neighbors (learns which friends matter more)
    """
    
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
    
    def forward(self, x, adj):
        """
        Args:
            x: [n_nodes, in_dim] node features
            adj: [n_nodes, n_nodes] adjacency matrix (with self-loops)
        Returns:
            [n_nodes, out_dim] updated node features
        """
        # Normalize adjacency matrix
        degree = adj.sum(dim=1, keepdim=True).clamp(min=1)
        adj_normalized = adj / degree
        
        # Aggregate neighbor features (message passing)
        aggregated = torch.mm(adj_normalized, x)
        
        # Transform
        return self.linear(aggregated)


class GCNForLinkPrediction(nn.Module):
    """
    GCN model for link (edge) prediction in PYMK.
    
    Architecture:
    Node features -> GCN Layer 1 -> ReLU -> GCN Layer 2 -> Node embeddings
    Edge prediction: dot_product(embedding_u, embedding_v) -> sigmoid
    """
    
    def __init__(self, input_dim, hidden_dim=32, embed_dim=16):
        super().__init__()
        self.gcn1 = SimpleGCNLayer(input_dim, hidden_dim)
        self.gcn2 = SimpleGCNLayer(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x, adj):
        """Get node embeddings."""
        h = F.relu(self.gcn1(x, adj))
        h = self.dropout(h)
        h = self.gcn2(h, adj)
        return h
    
    def predict_edge(self, x, adj, u, v):
        """Predict probability of edge between nodes u and v."""
        embeddings = self.forward(x, adj)
        score = (embeddings[u] * embeddings[v]).sum()
        return torch.sigmoid(score)


# Create node features (one-hot cluster + degree + random features)
n_nodes = G.number_of_nodes()
n_features = 8

node_features = np.zeros((n_nodes, n_features))
for n in range(n_nodes):
    node_features[n, cluster_labels[n]] = 1.0  # One-hot cluster
    node_features[n, 4] = G.degree(n) / 20.0   # Normalized degree
    node_features[n, 5] = random.random()        # Random feature 1
    node_features[n, 6] = random.random()        # Random feature 2
    node_features[n, 7] = random.random()        # Random feature 3

# Create adjacency matrix with self-loops
adj = nx.to_numpy_array(G)
adj = adj + np.eye(n_nodes)  # Add self-loops

x_tensor = torch.FloatTensor(node_features)
adj_tensor = torch.FloatTensor(adj)

# Instantiate and run
gcn_model = GCNForLinkPrediction(input_dim=n_features, hidden_dim=32, embed_dim=16)

with torch.no_grad():
    node_embeds = gcn_model(x_tensor, adj_tensor)

print(f"GCN Model Summary")
print(f"=" * 40)
print(f"Input features per node: {n_features}")
print(f"Hidden dim: 32")
print(f"Embedding dim: 16")
print(f"Output: {node_embeds.shape} node embeddings")
print(f"\nTo predict if users u and v will connect:")
print(f"  P(edge) = sigmoid(dot(embed_u, embed_v))")
print(f"\nExample: P(edge between user 0 and user 1):")
with torch.no_grad():
    p = gcn_model.predict_edge(x_tensor, adj_tensor, 0, 1)
    print(f"  {p.item():.4f} (before training -- random weights)")

In [ ]:
# === Train the GCN for link prediction ===

from sklearn.metrics import roc_auc_score

# Create training data: positive edges (existing) and negative edges (non-existing)
edges = list(G.edges())
non_edges = []
all_nodes = list(G.nodes())
while len(non_edges) < len(edges):
    u, v = random.choice(all_nodes), random.choice(all_nodes)
    if u != v and not G.has_edge(u, v) and (u, v) not in non_edges:
        non_edges.append((u, v))

# Training
optimizer = torch.optim.Adam(gcn_model.parameters(), lr=0.01)

print("Training GCN for Link Prediction...")
print(f"Positive edges: {len(edges)}, Negative edges: {len(non_edges)}")

for epoch in range(50):
    gcn_model.train()
    optimizer.zero_grad()
    
    node_embeds = gcn_model(x_tensor, adj_tensor)
    
    # Positive scores
    pos_scores = []
    for u, v in random.sample(edges, min(200, len(edges))):
        score = (node_embeds[u] * node_embeds[v]).sum()
        pos_scores.append(score)
    
    # Negative scores
    neg_scores = []
    for u, v in random.sample(non_edges, min(200, len(non_edges))):
        score = (node_embeds[u] * node_embeds[v]).sum()
        neg_scores.append(score)
    
    pos_tensor = torch.stack(pos_scores)
    neg_tensor = torch.stack(neg_scores)
    
    # BPR-style loss: push positive scores above negative
    labels = torch.cat([torch.ones(len(pos_scores)), torch.zeros(len(neg_scores))])
    scores = torch.cat([pos_tensor, neg_tensor])
    loss = F.binary_cross_entropy_with_logits(scores, labels)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        gcn_model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(scores).numpy()
            auc = roc_auc_score(labels.numpy(), preds)
        print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | AUC: {auc:.4f}")

print(f"\nGCN learned to distinguish existing edges from non-edges!")

## 5. Community Detection

Community detection finds groups of densely-connected nodes (like departments in a company or friend groups at school). This is useful for PYMK because people within the same community are more likely to connect.

### Algorithms:
- **Louvain:** Greedy modularity optimization (fast, widely used)
- **Label Propagation:** Nodes adopt the majority label of neighbors (very fast)
- **Spectral Clustering:** Uses eigenvalues of the graph Laplacian

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities, label_propagation_communities

# Louvain-like community detection (greedy modularity)
detected_communities = list(greedy_modularity_communities(G))

# Label propagation
lp_communities = list(label_propagation_communities(G))

print(f"Community Detection Results")
print(f"=" * 50)
print(f"True clusters: 4 departments")
print(f"\nGreedy Modularity:")
print(f"  Detected {len(detected_communities)} communities")
for i, comm in enumerate(detected_communities):
    # Check which true cluster dominates
    cluster_counts = defaultdict(int)
    for node in comm:
        cluster_counts[G.nodes[node]['cluster_name']] += 1
    dominant = max(cluster_counts, key=cluster_counts.get)
    purity = cluster_counts[dominant] / len(comm)
    print(f"  Community {i}: {len(comm)} nodes, dominant={dominant} (purity={purity:.0%})")

print(f"\nLabel Propagation:")
print(f"  Detected {len(lp_communities)} communities")

# Visualize detected vs true communities
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# True communities
true_colors = [colors[cluster_labels[n]] for n in G.nodes()]
nx.draw(G, pos, ax=axes[0], node_color=true_colors, node_size=100,
        edge_color='#cccccc', width=0.3)
axes[0].set_title('True Communities (departments)', fontsize=13)

# Detected communities
det_colors_map = {}
all_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE']
for i, comm in enumerate(detected_communities):
    for node in comm:
        det_colors_map[node] = all_colors[i % len(all_colors)]
det_colors = [det_colors_map.get(n, '#cccccc') for n in G.nodes()]
nx.draw(G, pos, ax=axes[1], node_color=det_colors, node_size=100,
        edge_color='#cccccc', width=0.3)
axes[1].set_title(f'Detected Communities ({len(detected_communities)} found)', fontsize=13)

plt.tight_layout()
plt.show()

print("Community detection is useful as an additional feature:")
print("  same_community(u, v) is a strong signal for connection prediction.")

## Interview Cheat Sheet: Graph Approaches

### Key Phrases to Drop

- "We formulate PYMK as **edge prediction** on the social graph, which captures neighborhood context that pointwise LTR cannot."
- "**Common neighbors** is the single strongest feature -- Meta found that 92% of new connections come from friends-of-friends."
- "For embeddings, I'd use **Node2Vec** with tuned p and q parameters, or **GraphSAGE** for scalability since it supports mini-batch training via neighborhood sampling."
- "GNNs learn from **both graph structure and node features**, combining the strengths of hand-crafted graph features and deep learning."
- "For link prediction, the score between two nodes is the **dot product of their GNN embeddings**, passed through a sigmoid."

### Model Comparison

| Approach | Uses Features? | Scales? | Handles New Nodes? | Best For |
|----------|---------------|---------|-------------------|----------|
| Hand-crafted (CN, Jaccard) | No (structure only) | Yes | Yes | Baseline |
| Node2Vec | No (structure only) | Moderate | No (retrain) | Medium graphs |
| GCN | Yes | Moderate | Yes (inductive) | Full-batch |
| GraphSAGE | Yes | Yes (sampling) | Yes (inductive) | Production |

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)